[![Buka di Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kang-prayit/visi-komputer-kesehatan/blob/main/notebooks/bab13.ipynb)

# Bab 13 — Dari Notebook ke Aplikasi: ONNX, Kuantisasi, Latensi CPU, dan Demo Gradio
**Buku:** *Visi Komputer untuk Kesehatan: Klasifikasi, Regresi, Deteksi, dan Segmentasi Citra Medis dengan Python*
**Penulis:** Prayitno, Karisma Trinanda Putra, Eko Prasetyo, Sunardi, Liliek Triyono

Jalankan di **Google Colab dengan GPU T4**, lalu **Runtime → Run all**.

Isi:
1. Latih cepat dua model PneumoniaMNIST 128 px (CC BY 4.0): ResNet-18 dan MobileNetV3-Small (pra-latih ImageNet).
2. Simpan/muat `state_dict`; **ekspor ONNX** (pra-pemrosesan normalisasi ikut di dalam graf) dan verifikasi keluaran dengan **ONNX Runtime**.
3. **Kuantisasi INT8** ONNX Runtime: dinamis (bobot) dan statis QDQ (kalibrasi 256 citra latih) — ukuran berkas, AUC, kesepakatan prediksi.
4. **Latensi CPU** (batch 1; 1 dan 2 *thread*): PyTorch FP32 vs ONNX FP32 vs ONNX INT8.
5. **Demo Gradio** untuk riset/edukasi — dijalankan dengan `share=False` (**tidak** membuat tautan publik), disertai peringatan "bukan alat diagnosis".
6. Artefak siap-bagi: `model_card.md`, `requirements.txt`, `app.py`.

Hasil → `/content/hasil_bab13/` → `/content/hasil_bab13.zip`.

## 0 Sel standar: mount Google Drive dan `pastikan_dataset()`

In [ ]:
# ===== SEL STANDAR BUKU: Drive + pastikan_dataset() (sama di semua notebook) =====
import os, sys, time, json, hashlib, shutil, zipfile, re, subprocess
from pathlib import Path
try:
    import requests, certifi
except ImportError:  # sudah ada di Colab; dipasang bila notebook dijalankan di tempat lain
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests", "certifi"], check=True)
    import requests, certifi
DI_COLAB = "google.colab" in sys.modules
if DI_COLAB:
    from google.colab import drive
    # Lokasi folder buku di Drive dapat diubah lewat variabel lingkungan BUKU_DRIVE_SUBPATH
    BUKU_DRIVE_SUBPATH = os.environ.get("BUKU_DRIVE_SUBPATH", "MyDrive/Buku Visi Komputer Kesehatan")
    for _coba in range(3):  # popup izin Drive kadang lambat: ulangi, dengan force_remount
        try:
            drive.mount("/content/drive", force_remount=_coba > 0); break
        except Exception as e:
            print("drive.mount gagal (", e, "), mencoba lagi ..."); time.sleep(5)
    AKAR_BUKU = Path("/content/drive") / BUKU_DRIVE_SUBPATH
    LOKAL = Path("/content/data_lokal")
else:  # dijalankan di luar Colab: arahkan ke folder lain lewat variabel lingkungan
    AKAR_BUKU = Path(os.environ.get("BUKU_DRIVE", str(Path.home() / "Buku Visi Komputer Kesehatan")))
    LOKAL = Path(os.environ.get("BUKU_LOKAL", "/tmp/data_lokal"))
DATASET_DIR = AKAR_BUKU / "dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True); LOKAL.mkdir(parents=True, exist_ok=True)
print("Folder dataset di Drive:", DATASET_DIR)

# Katalog sumber resmi: nama berkas -> [URL, ukuran byte, MD5]. MD5 None = sumber tidak menyediakan
# (MD5 malaria/kvasir dicatat penulis dari unduhan 27-10-2026; bila sumber memperbarui berkas, cek ukuran yang dipakai).
MEDMNIST = {  # Zenodo 10.5281/zenodo.10519652 (CC BY 4.0 kecuali DermaMNIST CC BY-NC 4.0)
    'adrenalmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/adrenalmnist3d.npz/content', 276833, 'bbd3c5a5576322bc4cdfea780653b1ce'],
    'adrenalmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/adrenalmnist3d_64.npz/content', 1880591, '17721accfe9fb005146a47d33bc54b2f'],
    'bloodmnist.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist.npz/content', 35461855, '7053d0359d879ad8a5505303e11de1dc'],
    'bloodmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_128.npz/content', 569072780, 'adace1e0ed228fccda1f39692059dd4c'],
    'bloodmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_224.npz/content', 1540731655, 'b718ff6835fcbdb22ba9eacccd7b2601'],
    'bloodmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_64.npz/content', 156251765, '2b94928a2ae4916078ca51e05b6b800b'],
    'breastmnist.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist.npz/content', 559580, '750601b1f35ba3300ea97c75c52ff8f6'],
    'breastmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_128.npz/content', 10961822, '363e4b3f8d712e9b5de15470a2aaadf1'],
    'breastmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_224.npz/content', 30903564, 'b56378a6eefa9fed602bb16d192d4c8b'],
    'breastmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_64.npz/content', 2846041, '742edef2a1fd1524b2efff4bd7ba9364'],
    'chestmnist.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist.npz/content', 82802576, '02c8a6516a18b556561a56cbdd36c4a8'],
    'chestmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_128.npz/content', 1426132371, 'db107e5590b27930b62dbaf558aebee3'],
    'chestmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_224.npz/content', 3889293042, '45bd33e6f06c3e8cdb481c74a89152aa'],
    'chestmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_64.npz/content', 401604127, '9de6cd0b934ebb5b7426cfba5efbae16'],
    'dermamnist.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist.npz/content', 19725078, '0744692d530f8e62ec473284d019b0c7'],
    'dermamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_128.npz/content', 372579947, '2defd784463fa5243564e855ed717de1'],
    'dermamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_224.npz/content', 1091112502, '8974907d8e169bef5f5b96bc506ae45d'],
    'dermamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_64.npz/content', 100130190, 'b70a2f5635c6199aeaa28c31d7202e1f'],
    'fracturemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/fracturemnist3d.npz/content', 3278419, '6aa7b0143a6b42da40027a9dda61302f'],
    'fracturemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/fracturemnist3d_64.npz/content', 26565236, 'f01d7e6316aedf4210da0da5b7437b42'],
    'nodulemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/nodulemnist3d.npz/content', 29299364, '8755a7e9e05a4d9ce80a24c3e7a256f3'],
    'nodulemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/nodulemnist3d_64.npz/content', 289765731, 'c47c5b7d457bf6332200d2ea6d64ecd8'],
    'octmnist.npz': ['https://zenodo.org/api/records/10519652/files/octmnist.npz/content', 54938180, 'c68d92d5b585d8d81f7112f81e2d0842'],
    'octmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_128.npz/content', 1297741899, '0a97e76651ace45c5d943ee3f65b63ae'],
    'octmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_224.npz/content', 3959197818, 'abc493b6d529d5de7569faaef2773ba3'],
    'octmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_64.npz/content', 311755777, 'e229e9440236b774d9f0dfef9d07bdaf'],
    'organamnist.npz': ['https://zenodo.org/api/records/10519652/files/organamnist.npz/content', 38247708, '68e3f8846a6bd62f0c9bf841c0d9eacc'],
    'organamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_128.npz/content', 707925571, 'eeae80d0a227a8d099027e1b3cfd3b60'],
    'organamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_224.npz/content', 1803859544, '50747347e05c87dd3aaf92c49f9f3170'],
    'organamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_64.npz/content', 200383427, '2dcccc29b88e6da5a01161ef20cda288'],
    'organcmnist.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist.npz/content', 15526411, 'b9ceb9546e10131b32923c5bbeaea2b1'],
    'organcmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_128.npz/content', 287499235, '773c1f009daa3fe5d9a2a201b2a7ed94'],
    'organcmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_224.npz/content', 760231860, '050f5e875dc056f6768abf94ec9995d1'],
    'organcmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_64.npz/content', 80344589, '3ce34a8724ea6f548e6db4744d03b6a9'],
    'organmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/organmnist3d.npz/content', 32657349, 'a0c5a1ff56af4f155c46d46fbb45a2fe'],
    'organmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/organmnist3d_64.npz/content', 361470105, '58a2205adf14a9d0a189cb06dc78bf10'],
    'organsmnist.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist.npz/content', 16528359, '9ab87b696fb54e2a387ebe992d6ed5f1'],
    'organsmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_128.npz/content', 305188399, 'ded0c5fa01a95dc4978b956f613e9b8e'],
    'organsmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_224.npz/content', 802713625, 'b354719e553fbbb2513d5533f52a4cb1'],
    'organsmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_64.npz/content', 85866166, '53a6d115339d874c25e309a994ff46d3'],
    'pathmnist.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist.npz/content', 205615438, 'a8b06965200029087d5bd730944a56c1'],
    'pathmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_128.npz/content', 4257786775, 'ac42d08fb904d92c244187169d1fd1d9'],
    'pathmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_224.npz/content', 12629854322, '2c51a510bcdc9cf8ddb2af93af1eadec'],
    'pathmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_64.npz/content', 1069422170, '55aa9c1e0525abe5a6b9d8343a507616'],
    'pneumoniamnist.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist.npz/content', 4170669, '28209eda62fecd6e6a2d98b1501bb15f'],
    'pneumoniamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_128.npz/content', 75506212, '05b46931834c231683c68f40c47b2971'],
    'pneumoniamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_224.npz/content', 214384716, 'd6a3c71de1b945ea11211b03746c1fe1'],
    'pneumoniamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_64.npz/content', 20606998, '8f4eceb4ccffa70c672198ea285246c6'],
    'retinamnist.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist.npz/content', 3291041, 'bd4c0672f1bba3e3a89f0e4e876791e4'],
    'retinamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_128.npz/content', 46467058, 'e48e916a24454daf90583d4e6efb1a18'],
    'retinamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_224.npz/content', 127992567, 'eae7e3b6f3fcbda4ae613ebdcbe35348'],
    'retinamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_64.npz/content', 13200879, 'afda852cc34dcda56f86ad2b2457dbcc'],
    'synapsemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/synapsemnist3d.npz/content', 38034583, '1235b78a3cd6280881dd7850a78eadb6'],
    'synapsemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/synapsemnist3d_64.npz/content', 452767391, '43bd14ebf3af9d3dd072446fedc14d5e'],
    'tissuemnist.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist.npz/content', 124962739, 'ebe78ee8b05294063de985d821c1c34b'],
    'tissuemnist_128.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_128.npz/content', 1625223910, '61b955355d7425a89687b06cca3ce0c2'],
    'tissuemnist_224.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_224.npz/content', 3433703243, 'b077128c4a949f0a4eb01517f9037b9c'],
    'tissuemnist_64.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_64.npz/content', 555266078, '123ece2eba09d0aa5d698fda57103344'],
    'vesselmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/vesselmnist3d.npz/content', 398349, 'b41fd4f7e7e2feedddb201585ecafa1b'],
    'vesselmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/vesselmnist3d_64.npz/content', 2653988, '6bb274a8846e1097066dcd64e2c4520f']
}
HC18 = {      # Zenodo 10.5281/zenodo.1327317 (CC BY 4.0)
    'test_set.zip': ['https://zenodo.org/api/records/1327317/files/test_set.zip/content', 43518463, '8402af5d137ef40a2888c1011ef3fe7e'],
    'test_set_pixel_size.csv': ['https://zenodo.org/api/records/1327317/files/test_set_pixel_size.csv/content', 9096, '8476dc198cc6542f26c57e87faeee033'],
    'training_set.zip': ['https://zenodo.org/api/records/1327317/files/training_set.zip/content', 132926838, '00eb8198b9a505b2b3a6dfc740382497'],
    'training_set_pixel_size_and_HC.csv': ['https://zenodo.org/api/records/1327317/files/training_set_pixel_size_and_HC.csv/content', 33688, 'c0761518fece2bd2d2ad4218f2cd9777']
}
KATALOG = {
    "medmnist":   {"berkas": MEDMNIST},
    "hc18":       {"berkas": HC18},
    "malaria":    {"berkas": {"cell_images.zip": ["https://data.lhncbc.nlm.nih.gov/public/Malaria/cell_images.zip", 353452851, "2f2e94a91d413e5b0bab6342fddb6de6"]},
                   "isi": {"cell_images.zip": {".png": 27558}}},
    "kvasir_seg": {"berkas": {"kvasir-seg.zip": ["https://datasets.simula.no/downloads/kvasir-seg.zip", 46227172, "6323d9094df93b35d43069a566ee1ca3"]},
                   "isi": {"kvasir-seg.zip": {".jpg": 2000}}},
    "bccd":       {"berkas": {"BCCD_Dataset-master.zip": ["https://github.com/Shenggan/BCCD_Dataset/archive/refs/heads/master.zip", None, None]},
                   "isi": {"BCCD_Dataset-master.zip": {"/JPEGImages/.jpg": 364, "/Annotations/.xml": 364}}},
}
# Rantai sertifikat: server datasets.simula.no tidak mengirim sertifikat perantara (Sectigo).
# Kita TIDAK mematikan verifikasi TLS; sertifikat perantara publik diambil dari crt.sh dan ditambahkan ke bundel CA.
CRT_PERANTARA = {"datasets.simula.no": "https://crt.sh/?d=4267304690"}

def md5_berkas(p, blok=8 << 20):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(blok), b""): h.update(b)
    return h.hexdigest()

def _cek_isi_zip(p, aturan):
    nama = zipfile.ZipFile(p).namelist()
    for pola, n in aturan.items():
        folder, ext = (pola.rsplit("/", 1)[0] + "/", pola.rsplit("/", 1)[1]) if "/" in pola else ("", pola)
        jml = sum(1 for x in nama if x.lower().endswith(ext) and folder in x and "__MACOSX" not in x)
        if jml != n: return f"isi zip {pola}: {jml} berkas, seharusnya {n}"
    return None

def berkas_lengkap(nama, fn):
    """Cek berkas di Drive: ada, ukuran sesuai, MD5 sesuai (MD5 dihitung sekali lalu dicatat di <fn>.md5ok)."""
    url, ukuran, md5 = KATALOG[nama]["berkas"][fn]
    p = DATASET_DIR / nama / fn
    if not p.exists(): return False, "belum ada"
    if ukuran and p.stat().st_size != ukuran: return False, f"ukuran {p.stat().st_size} B, seharusnya {ukuran} B"
    tanda = p.with_name(p.name + ".md5ok")
    if md5 and not (tanda.exists() and tanda.read_text().strip() == md5):
        m = md5_berkas(p)
        if m != md5: return False, f"MD5 {m} tidak cocok ({md5})"
        tanda.write_text(md5)
    aturan = KATALOG[nama].get("isi", {}).get(fn)
    if aturan:
        tanda_isi = p.with_name(p.name + ".isiok")
        if not tanda_isi.exists():
            salah = _cek_isi_zip(p, aturan)
            if salah: return False, salah
            tanda_isi.write_text(json.dumps(aturan))
    return True, "lengkap"

def _bundel_ca(host):
    import certifi, requests
    pem = requests.get(CRT_PERANTARA[host], timeout=60).text
    assert "BEGIN CERTIFICATE" in pem
    b = LOKAL / f"ca_{host}.pem"; b.write_text(open(certifi.where()).read() + "\n" + pem); return str(b)

def unduh(url, tujuan):
    import requests
    from urllib.parse import urlparse
    tmp = LOKAL / "_unduh" / tujuan.name; tmp.parent.mkdir(parents=True, exist_ok=True)
    host = urlparse(url).hostname; verify = True
    for coba in range(2):
        try:
            with requests.get(url, stream=True, timeout=120, verify=verify) as r:
                r.raise_for_status(); total = int(r.headers.get("content-length", 0)); n = 0; t0 = time.time()
                with open(tmp, "wb") as f:
                    for b in r.iter_content(4 << 20):
                        f.write(b); n += len(b)
                        if time.time() - t0 > 15: print(f"   ... {n/1e6:.0f}/{total/1e6:.0f} MB"); t0 = time.time()
            break
        except requests.exceptions.SSLError:
            if host in CRT_PERANTARA and verify is True:
                print("   rantai sertifikat server tidak lengkap; menambahkan sertifikat perantara resmi ke bundel CA")
                verify = _bundel_ca(host)
            else:
                raise
    tujuan.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(tmp, tujuan)          # salin ke Drive (permanen)
    return tmp                            # salinan lokal tetap ada untuk dipakai sesi ini

def pastikan_dataset(nama, berkas=None):
    """Pastikan dataset ada & lengkap di Drive; bila belum, unduh dari sumber resmi ke Drive. Kembalikan folder Drive-nya."""
    daftar = berkas or list(KATALOG[nama]["berkas"])
    for fn in daftar:
        ok, alasan = berkas_lengkap(nama, fn)
        if ok:
            print(f"Dataset {nama}/{fn} ditemukan di Drive ({(DATASET_DIR/nama/fn).stat().st_size/1e6:.1f} MB).")
            continue
        print(f"Dataset {nama}/{fn} belum ada atau tidak lengkap ({alasan}); mengunduh dari sumber resmi...")
        t0 = time.time(); unduh(KATALOG[nama]["berkas"][fn][0], DATASET_DIR / nama / fn)
        for t in (".md5ok", ".isiok"):
            q = DATASET_DIR / nama / (fn + t)
            if q.exists(): q.unlink()
        ok, alasan = berkas_lengkap(nama, fn)
        if not ok: raise RuntimeError(f"{nama}/{fn} gagal diverifikasi setelah diunduh: {alasan}")
        print(f"   selesai {time.time()-t0:.0f} s, tersimpan di Drive dan terverifikasi.")
    return DATASET_DIR / nama

def ekstrak_lokal(nama, fn):
    """Ekstrak arsip dari Drive ke disk lokal Colab (/content), sekali per sesi.
    Alasan: membaca ribuan berkas kecil langsung dari Drive (FUSE) sangat lambat; disk lokal jauh lebih cepat."""
    tujuan = LOKAL / nama; tanda = tujuan / (".ekstrak_" + fn)
    if tanda.exists(): print(f"{fn} sudah diekstrak di {tujuan}"); return tujuan
    t0 = time.time(); src = LOKAL / "_unduh" / fn
    if not src.exists() or src.stat().st_size != (DATASET_DIR / nama / fn).stat().st_size:
        src.parent.mkdir(parents=True, exist_ok=True); shutil.copyfile(DATASET_DIR / nama / fn, src)
    with zipfile.ZipFile(src) as z: z.extractall(tujuan)
    tanda.write_text("ok"); print(f"{fn} diekstrak ke {tujuan} dalam {time.time()-t0:.0f} s"); return tujuan

def berkas_medmnist(flag, size):
    return f"{flag}.npz" if size == 28 else f"{flag}_{size}.npz"
# ===== akhir sel standar =====

## 13.0 Lingkungan

In [ ]:
import os, sys, time, json, platform, subprocess, contextlib, shutil, random
from pathlib import Path
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # perlu untuk algoritme deterministik di GPU
T_MULAI = time.perf_counter()
SMOKE = os.environ.get("BAB13_SMOKE") == "1"          # uji cepat penulis di luar Colab (CPU, subset kecil); di Colab = False
OUT = Path(os.environ.get("BAB13_OUT", "/content/hasil_bab13"))
if OUT.exists(): shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)

WAKTU = {}
@contextlib.contextmanager
def catat_waktu(nama):
    t0 = time.perf_counter()
    yield
    WAKTU[nama] = round(time.perf_counter() - t0, 2)
    print(f"[waktu] {nama}: {WAKTU[nama]:.1f} s")

def jalankan(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60).stdout
    except Exception as e:
        return f"(gagal: {e})"

with catat_waktu("0_info_lingkungan"):
    smi = jalankan("nvidia-smi")
    print(smi if smi.strip() else "nvidia-smi tidak tersedia (tidak ada GPU?). Di Colab: Runtime > Change runtime type > T4 GPU.")
    info_cpu = jalankan("lscpu | grep -E 'Model name|^CPU\\(s\\)'"); info_mem = jalankan("free -h | head -2"); info_disk = jalankan("df -h /content 2>/dev/null || df -h /")
    print(info_cpu, info_mem, info_disk)
    (OUT / "info_lingkungan.txt").write_text(f"Python: {sys.version}\nPlatform: {platform.platform()}\n\n{smi}\n{info_cpu}\n{info_mem}\n{info_disk}")
print("SMOKE =", SMOKE, "| OUT =", OUT)

In [ ]:
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"          # matikan telemetri Gradio
with catat_waktu("1_pip_install"):
    # pip langsung (bukan jalankan(): batas 60 s di sana dapat memutus pemasangan gradio/onnx di tengah jalan)
    r_pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "medmnist==3.0.2", "onnx", "onnxruntime", "onnxscript", "gradio"], capture_output=True, text=True, timeout=900)
    print("pip selesai, kode keluar", r_pip.returncode); print(r_pip.stderr[-2000:]) if r_pip.returncode else None
    assert r_pip.returncode == 0, "pip install gagal; lihat pesan di atas"
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib, matplotlib.ticker; import matplotlib.pyplot as plt
import torchvision, sklearn, medmnist
from medmnist import INFO
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 9})
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VERSI = {"python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
         "numpy": np.__version__, "scikit-learn": sklearn.__version__, "medmnist": medmnist.__version__,
         "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}
print(json.dumps(VERSI, indent=1))
def npz_medmnist(flag, size):
    fn = berkas_medmnist(flag, size); folder = pastikan_dataset("medmnist", [fn]); return np.load(folder / fn)
SEED = 42
def seed_semua(seed=SEED):
    """Seed Python, NumPy, PyTorch (CPU+GPU) dan minta algoritme deterministik."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
seed_semua()
# ---------- Perangkat latih bersama (sama di notebook bab 04 dst.) ----------
MEAN_IN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); SD_IN = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
AMP = DEV.type == "cuda"

def ke_tensor_uint8(X):
    """(N,H,W) atau (N,H,W,C) uint8 -> tensor (N,C,H,W) uint8; disimpan di GPU bila ada (data kecil)."""
    t = torch.as_tensor(X)
    t = t[:, None] if t.ndim == 3 else t.permute(0, 3, 1, 2)
    return t.contiguous().to(DEV)

def augmentasi_gpu(x, flip_h=False, flip_v=False, rot90=False, rot_derajat=0.0, geser=0.0, kecerahan=0.1, kontras=0.1, warna=0.0):
    """x float (N,C,H,W) di [0,1]. Semua acak per sampel kecuali rot90 (per batch)."""
    n = x.shape[0]; d = x.device
    if flip_h: x = torch.where((torch.rand(n, device=d) < 0.5)[:, None, None, None], x.flip(3), x)
    if flip_v: x = torch.where((torch.rand(n, device=d) < 0.5)[:, None, None, None], x.flip(2), x)
    if rot90: x = torch.rot90(x, int(torch.randint(0, 4, (1,)).item()), (2, 3))
    if rot_derajat or geser:
        a = (torch.rand(n, device=d) * 2 - 1) * rot_derajat * np.pi / 180
        t = (torch.rand(n, 2, device=d) * 2 - 1) * geser * 2
        th = torch.stack([torch.stack([a.cos(), -a.sin(), t[:, 0]], 1), torch.stack([a.sin(), a.cos(), t[:, 1]], 1)], 1)
        x = F.grid_sample(x, F.affine_grid(th, x.shape, align_corners=False), padding_mode="border", align_corners=False)
    if warna and x.shape[1] == 3: x = x * (1 + (torch.rand(n, 3, 1, 1, device=d) * 2 - 1) * warna)
    m = x.mean((1, 2, 3), keepdim=True)
    x = ((x - m) * (1 + (torch.rand(n, 1, 1, 1, device=d) * 2 - 1) * kontras) + m) * (1 + (torch.rand(n, 1, 1, 1, device=d) * 2 - 1) * kecerahan)
    return x.clamp(0, 1)

class ResNet18(nn.Module):
    """ResNet-18 torchvision; masukan 1 atau 3 kanal [0,1], opsional diperbesar ke `uk` piksel, normalisasi ImageNet."""
    def __init__(self, k, pretrain=True, uk=None, p_drop=0.0):
        super().__init__(); self.uk = uk
        w = None
        if pretrain:
            w = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
            try: self.net = torchvision.models.resnet18(weights=w)
            except Exception as e:
                if not SMOKE: raise
                print("SMOKE: bobot ImageNet gagal diunduh, pakai bobot acak:", e); self.net = torchvision.models.resnet18(weights=None)
        else: self.net = torchvision.models.resnet18(weights=None)
        self.net.fc = nn.Sequential(nn.Dropout(p_drop), nn.Linear(self.net.fc.in_features, k))
    def forward(self, x):
        if x.shape[1] == 1: x = x.repeat(1, 3, 1, 1)
        if self.uk and x.shape[-1] != self.uk: x = F.interpolate(x, size=(self.uk, self.uk), mode="bilinear", align_corners=False)
        return self.net((x - MEAN_IN.to(x.device)) / SD_IN.to(x.device))

@torch.no_grad()
def logit_semua(model, X, bs=512, praproses=None):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        x = X[i:i + bs].to(DEV).float().div(255)
        if praproses: x = praproses(x)
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP): out.append(model(x).float())
    return torch.cat(out)

def latih(nama, model, X, Y, Xv, Yv, fungsi_loss, skor_val, maks_epoch=30, patience=5, aug=None, lr=1e-3, bs=128,
          tahap=None, bobot_sampel=None, praproses=None, seed=SEED, wd=1e-4):
    """Latih dengan AMP; early stopping memantau skor_val(logit_val, y_val) (lebih tinggi lebih baik).
    tahap: daftar (n_epoch, lr, beku_backbone). bobot_sampel: oversampling (sampling dengan pengembalian)."""
    seed_semua(seed); model = model.to(DEV); Yd = torch.as_tensor(Y).to(DEV); Yvd = torch.as_tensor(Yv).to(DEV)
    if SMOKE: maks_epoch, patience = min(maks_epoch, 2), 1
    scaler = torch.amp.GradScaler("cuda", enabled=AMP); log = []; best = (-np.inf, None, 0); ep_g = 0
    g = torch.Generator(device="cpu").manual_seed(seed)
    for n_ep, lr_t, beku in (tahap or [(maks_epoch, lr, False)]):
        if SMOKE: n_ep = min(n_ep, 2)
        for nm, p in model.named_parameters(): p.requires_grad = (not beku) or nm.startswith("net.fc")
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr_t, weight_decay=wd); tunggu = 0
        for ep in range(n_ep):
            model.train(); t0 = time.perf_counter()
            if bobot_sampel is not None: perm = torch.multinomial(torch.as_tensor(bobot_sampel, dtype=torch.float), len(Y), replacement=True, generator=g)
            else: perm = torch.randperm(len(Y), generator=g)
            perm = perm.to(X.device); tl = 0.0; n = 0
            for i in range(0, len(perm), bs):
                b = perm[i:i + bs]; x = X[b].to(DEV).float().div(255)
                if aug: x = augmentasi_gpu(x, **aug)
                if praproses: x = praproses(x)
                with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
                    lg = model(x)
                loss = fungsi_loss(lg.float(), Yd[b.to(DEV)])
                opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                tl += loss.item() * len(b); n += len(b)
            if DEV.type == "cuda": torch.cuda.synchronize()
            dt = time.perf_counter() - t0; ep_g += 1
            lv = logit_semua(model, Xv, praproses=praproses); vl = fungsi_loss(lv, Yvd).item(); sv = float(skor_val(lv, Yv))
            log.append({"model": nama, "epoch": ep_g, "tahap_beku": beku, "loss_latih": tl / n, "loss_val": vl, "skor_val": sv, "detik_epoch": dt})
            print(f"{nama} ep{ep_g:02d} loss {tl/n:.4f}/{vl:.4f} skor_val {sv:.4f} {dt:.1f}s")
            if sv > best[0]: best = (sv, {k: v.detach().clone() for k, v in model.state_dict().items()}, ep_g); tunggu = 0
            else:
                tunggu += 1
                if not beku and tunggu >= patience: print(f"  early stopping (patience {patience})"); break
    model.load_state_dict(best[1])
    return model, pd.DataFrame(log), best[2]

def n_param(m): return sum(p.numel() for p in m.parameters())

def ci_bootstrap(fn, y, p, n_boot=1000, seed=SEED, strata=True):
    """CI 95 % persentil; resampling bertingkat per kelas agar setiap sampel memuat semua kelas."""
    r = np.random.default_rng(seed); y = np.asarray(y); idx_k = [np.where(y == k)[0] for k in np.unique(y)]; v = []
    for _ in range(n_boot):
        i = np.concatenate([r.choice(ik, len(ik)) for ik in idx_k]) if strata else r.choice(len(y), len(y))
        try: v.append(fn(y[i], p[i]))
        except ValueError: pass
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

def gambar_kurva(LOG, judul_skor, path):
    k = len(LOG); fig, ax = plt.subplots(1, k, figsize=(4.2 * k, 3.2), squeeze=False); ax = ax[0]
    for a, lg in zip(ax, LOG):
        a.plot(lg.epoch, lg.loss_latih, "o-", ms=3, label="loss latih"); a.plot(lg.epoch, lg.loss_val, "s-", ms=3, label="loss val")
        a2 = a.twinx(); a2.plot(lg.epoch, lg.skor_val, "^-", ms=3, color="tab:green", label=judul_skor); a2.set_ylabel(judul_skor, color="tab:green")
        eb = int(lg.loc[lg.skor_val.idxmax(), "epoch"]); a.axvline(eb, color="red", ls="--", lw=1, label=f"terbaik ep {eb}")
        if lg.tahap_beku.any(): a.axvspan(0.5, lg[lg.tahap_beku].epoch.max() + 0.5, color="gray", alpha=0.15, label="tahap beku")
        a.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True)); a.set_title(lg.model.iloc[0], fontsize=8); a.set_xlabel("epoch"); a.legend(fontsize=6)
    plt.tight_layout(); plt.savefig(path, bbox_inches="tight"); plt.show()
from sklearn.metrics import roc_auc_score
import onnx, onnxruntime as ort, gradio as gr
VERSI.update({'onnx': onnx.__version__, 'onnxruntime': ort.__version__, 'gradio': gr.__version__}); print(VERSI)

## 13.1 Data dan dua model cepat

In [ ]:
UK = 28 if SMOKE else 128
with catat_waktu("2_muat_data"):
    d = npz_medmnist("pneumoniamnist", UK)
    D = {s: (d[f"{s}_images"], d[f"{s}_labels"].reshape(-1).astype(np.int64)) for s in ("train", "val", "test")}
    if SMOKE:
        r = np.random.default_rng(0); D = {s: (x[i], y[i]) for s, (x, y) in D.items() for i in [r.choice(len(y), min(len(y), 300), replace=False)]}
XT = {s: ke_tensor_uint8(D[s][0]) for s in D}; Y = {s: D[s][1] for s in D}; yt = Y["test"]
class MobileNetV3S(nn.Module):
    def __init__(self):
        super().__init__()
        try: self.net = torchvision.models.mobilenet_v3_small(weights=torchvision.models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        except Exception as e:
            if not SMOKE: raise
            print("SMOKE: bobot acak", e); self.net = torchvision.models.mobilenet_v3_small(weights=None)
        self.net.classifier[-1] = nn.Linear(self.net.classifier[-1].in_features, 1)
    def forward(self, x):
        x = x.repeat(1, 3, 1, 1); return self.net((x - MEAN_IN.to(x.device)) / SD_IN.to(x.device))
loss_bce = lambda lg, y: F.binary_cross_entropy_with_logits(lg[:, 0], y.float())
def skor_auc(lg, y): return roc_auc_score(y, lg[:, 0].cpu().numpy())
AUG = dict(rot_derajat=7, geser=0.05, kecerahan=0.1, kontras=0.1)
MODEL = {}
for nm, fm, tahap in [("ResNet-18", lambda: ResNet18(1, True), [(2, 1e-3, True), (8, 1e-4, False)]), ("MobileNetV3-Small", lambda: MobileNetV3S(), [(10, 1e-3, False)])]:
    with catat_waktu(f"latih_{nm}"):
        MODEL[nm] = latih(nm, fm(), XT["train"], Y["train"], XT["val"], Y["val"], loss_bce, skor_auc, patience=3, aug=AUG, tahap=tahap, bs=64, seed=0)[0]
    print(nm, "AUC uji (PyTorch, GPU):", roc_auc_score(yt, logit_semua(MODEL[nm], XT["test"])[:, 0].cpu().numpy()))

## 13.2 Menyimpan, memuat, dan mengekspor ke ONNX
Praktik baik: simpan **`state_dict`** (bukan seluruh objek) beserta metadata (kelas, ukuran masukan, normalisasi, versi pustaka).
Ekspor ONNX memakai masukan float32 `[batch, 1, UK, UK]` bernilai 0–1; normalisasi ImageNet sudah di dalam graf. Sumbu *batch* dibuat dinamis.

In [ ]:
X_UJI = XT["test"].float().div(255).cpu()
def ekspor(nm, model):
    model = model.float().cpu().eval(); torch.save({"state_dict": model.state_dict(), "kelas": ["normal", "pneumonia"], "ukuran": UK, "versi": VERSI}, OUT / f"{nm}.pt")
    contoh = torch.rand(1, 1, UK, UK); f = OUT / f"{nm}_fp32.onnx"
    try:
        torch.onnx.export(model, (contoh,), str(f), input_names=["citra"], output_names=["logit"], dynamic_axes={"citra": {0: "batch"}, "logit": {0: "batch"}}, opset_version=17, dynamo=False)
        jalur = "TorchScript (dynamo=False)"
    except Exception as e:
        print("Eksportir lama gagal, beralih ke dynamo:", repr(e)[:200])
        torch.onnx.export(model, (contoh,), str(f), input_names=["citra"], output_names=["logit"], dynamic_shapes={"x": {0: torch.export.Dim("batch", min=1, max=1024)}}, dynamo=True)
        jalur = "torch.export (dynamo=True)"
    onnx.checker.check_model(onnx.load(str(f))); return f, jalur
def sesi(f, thread=None):
    so = ort.SessionOptions()
    if thread: so.intra_op_num_threads = thread; so.inter_op_num_threads = 1
    return ort.InferenceSession(str(f), so, providers=["CPUExecutionProvider"])
def ort_logit(s, X, bs=256): return np.concatenate([s.run(None, {"citra": X[i:i + bs].numpy()})[0][:, 0] for i in range(0, len(X), bs)])
ONNX, baris = {}, []
with catat_waktu("3_ekspor_onnx"):
    for nm, m in MODEL.items():
        f, jalur = ekspor(nm, m); ONNX[nm] = {"FP32": f}
        with torch.no_grad(): lt = m(X_UJI[:256]).numpy()[:, 0]
        lo = ort_logit(sesi(f), X_UJI[:256])
        baris.append({"model": nm, "jalur_ekspor": jalur, "maks_|beda_logit|_torch_vs_onnx": float(np.abs(lt - lo).max()), "ukuran_MB": f.stat().st_size / 2 ** 20})
T131 = pd.DataFrame(baris); T131.to_csv(OUT / "tabel_13_1_ekspor_onnx.csv", index=False); T131

## 13.3 Kuantisasi INT8 dengan ONNX Runtime
- **Dinamis**: bobot dikuantisasi ke INT8, aktivasi dikuantisasi saat inferensi (tanpa data kalibrasi).
- **Statis QDQ**: skala aktivasi ditentukan dari **256 citra latih** (kalibrasi MinMax), bobot per kanal.
Model terkuantisasi dibandingkan dengan FP32: ukuran, AUC uji, dan persentase prediksi (ambang 0,5) yang sama.

In [ ]:
from onnxruntime.quantization import quantize_dynamic, quantize_static, QuantType, QuantFormat, CalibrationDataReader
from onnxruntime.quantization.shape_inference import quant_pre_process
class PembacaKalibrasi(CalibrationDataReader):
    def __init__(self, X, bs=32): self.it = iter([{"citra": X[i:i + bs].numpy()} for i in range(0, len(X), bs)])
    def get_next(self): return next(self.it, None)
r = np.random.default_rng(SEED); X_KAL = XT["train"][torch.as_tensor(r.choice(len(Y["train"]), min(256, len(Y["train"])), replace=False), device=XT["train"].device)].float().div(255).cpu()
baris = []
with catat_waktu("4_kuantisasi"):
    for nm in MODEL:
        f32 = ONNX[nm]["FP32"]; fpre = OUT / f"{nm}_pre.onnx"; quant_pre_process(str(f32), str(fpre))
        fdyn = OUT / f"{nm}_int8_dinamis.onnx"; quantize_dynamic(str(fpre), str(fdyn), weight_type=QuantType.QInt8)
        fst = OUT / f"{nm}_int8_statis.onnx"
        quantize_static(str(fpre), str(fst), PembacaKalibrasi(X_KAL), quant_format=QuantFormat.QDQ, per_channel=True, weight_type=QuantType.QInt8, activation_type=QuantType.QUInt8)
        ONNX[nm].update({"INT8 dinamis": fdyn, "INT8 statis": fst}); fpre.unlink()
        p32 = 1 / (1 + np.exp(-ort_logit(sesi(f32), X_UJI)))
        for versi, f in ONNX[nm].items():
            try: p = 1 / (1 + np.exp(-ort_logit(sesi(f), X_UJI))); galat = ""
            except Exception as e: p = np.full(len(yt), np.nan); galat = repr(e)[:120]
            baris.append({"model": nm, "versi": versi, "ukuran_MB": f.stat().st_size / 2 ** 20, "AUC_uji": roc_auc_score(yt, p) if not np.isnan(p).any() else np.nan,
                          "persen_prediksi_sama_dg_FP32": float(((p >= 0.5) == (p32 >= 0.5)).mean() * 100), "maks_|beda_prob|": float(np.nanmax(np.abs(p - p32))), "catatan": galat})
T132 = pd.DataFrame(baris); T132.to_csv(OUT / "tabel_13_2_kuantisasi.csv", index=False); T132

## 13.4 Latensi CPU (batch 1)
Median dari 100 inferensi setelah 10 pemanasan. CPU Colab biasanya 2 vCPU; angka ini **bukan** latensi perangkat tepi (mis. Raspberry Pi), tetapi
memperlihatkan pengaruh format dan kuantisasi. Informasi CPU tersimpan di `info_lingkungan.txt`.

In [ ]:
def ukur(fn, n=100, panas=10):
    for _ in range(panas): fn()
    t = []
    for _ in range(n): t0 = time.perf_counter(); fn(); t.append(time.perf_counter() - t0)
    return float(np.median(t) * 1000), float(np.percentile(t, 90) * 1000)
x1 = X_UJI[:1]; N = 10 if SMOKE else 100; baris = []
with catat_waktu("5_latensi"):
    for th in (1, 2):
        torch.set_num_threads(th)
        for nm, m in MODEL.items():
            m = m.float().cpu().eval()
            with torch.no_grad(): med, p90 = ukur(lambda: m(x1), n=N)
            baris.append({"model": nm, "runtime": "PyTorch FP32", "thread": th, "ms_median": med, "ms_p90": p90})
            for versi, f in ONNX[nm].items():
                try:
                    s = sesi(f, th); med, p90 = ukur(lambda: s.run(None, {"citra": x1.numpy()}), n=N)
                except Exception as e: med, p90 = np.nan, np.nan
                baris.append({"model": nm, "runtime": f"ONNX Runtime {versi}", "thread": th, "ms_median": med, "ms_p90": p90})
T133 = pd.DataFrame(baris); T133.to_csv(OUT / "tabel_13_3_latensi_cpu.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for a, th in zip(ax, (1, 2)):
    d_ = T133[T133.thread == th]; lab = [f"{a_}\n{b_}" for a_, b_ in zip(d_.model, d_.runtime.str.replace("ONNX Runtime ", "ORT "))]
    a.barh(range(len(d_)), d_.ms_median); a.set_yticks(range(len(d_))); a.set_yticklabels(lab, fontsize=6); a.set_xlabel("ms per citra (median)"); a.set_title(f"{th} thread CPU")
plt.tight_layout(); plt.savefig(OUT / "gambar_13_1_latensi.png", bbox_inches="tight"); plt.show(); T133

## 13.5 Demo Gradio (riset/edukasi, bukan klinis)
Antarmuka memuat model **ONNX** (tanpa PyTorch) sehingga sama dengan yang akan dipakai di `app.py`. Diluncurkan dengan **`share=False`**:
di Colab antarmuka tampil di bawah sel melalui *proxy* Colab dan **tidak** membuat tautan publik `*.gradio.live`.
Jangan mengunggah citra pasien sungguhan ke demo ini.

In [ ]:
PILIH_DEMO = max([n for n in MODEL], key=lambda n: T132[(T132.model == n) & (T132.versi == "FP32")].AUC_uji.iloc[0])
SESI_DEMO = sesi(ONNX[PILIH_DEMO]["FP32"])
PERINGATAN = ("⚠️ Demo riset/edukasi dari buku *Visi Komputer untuk Kesehatan*. BUKAN alat diagnosis dan tidak boleh dipakai untuk keputusan klinis. "
              "Model dilatih pada PneumoniaMNIST (X-ray dada anak, 128 px) dan dapat salah pada citra dari sumber lain.")
def prediksi_demo(img):
    if img is None: return {}
    from PIL import Image as _I
    im = _I.fromarray(np.asarray(img)).convert("L").resize((UK, UK), _I.BILINEAR)
    x = (np.asarray(im, dtype=np.float32) / 255.0)[None, None]
    p = float(1 / (1 + np.exp(-SESI_DEMO.run(None, {"citra": x})[0][0, 0])))
    return {"pneumonia": p, "normal": 1 - p}
CONTOH = []
(OUT / "contoh_demo").mkdir(exist_ok=True)
from PIL import Image as _I
for k, i in enumerate([int(np.where(yt == 0)[0][0]), int(np.where(yt == 1)[0][0])]):
    fn = OUT / "contoh_demo" / f"uji_{i}_{['normal', 'pneumonia'][yt[i]]}.png"; _I.fromarray(D["test"][0][i]).save(fn); CONTOH.append([str(fn)])
print({c[0]: prediksi_demo(np.array(_I.open(c[0]))) for c in CONTOH})
demo = gr.Interface(fn=prediksi_demo, inputs=gr.Image(type="numpy", image_mode="L", label="X-ray dada (PNG/JPG)"), outputs=gr.Label(num_top_classes=2, label="probabilitas"),
                    title=f"Demo edukasi: pneumonia pada X-ray dada ({PILIH_DEMO}, ONNX)", description=PERINGATAN, examples=CONTOH, flagging_mode="never")
demo.launch(share=False, prevent_thread_lock=True, quiet=True, **({"server_name": "127.0.0.1"} if SMOKE else {}))
if SMOKE: demo.close()

## 13.6 Artefak siap-bagi: `app.py`, `requirements.txt`, `model_card.md`

In [ ]:
(OUT / "app.py").write_text(f'''# Demo edukasi Gradio — Bab 13 buku Visi Komputer untuk Kesehatan. BUKAN alat diagnosis.
import numpy as np, onnxruntime as ort, gradio as gr
from PIL import Image
UK = {UK}
SESI = ort.InferenceSession("{PILIH_DEMO}_fp32.onnx", providers=["CPUExecutionProvider"])
PERINGATAN = {PERINGATAN!r}
def prediksi(img):
    if img is None: return {{}}
    x = (np.asarray(Image.fromarray(np.asarray(img)).convert("L").resize((UK, UK), Image.BILINEAR), dtype=np.float32) / 255.0)[None, None]
    p = float(1 / (1 + np.exp(-SESI.run(None, {{"citra": x}})[0][0, 0])))
    return {{"pneumonia": p, "normal": 1 - p}}
gr.Interface(fn=prediksi, inputs=gr.Image(type="numpy", image_mode="L"), outputs=gr.Label(num_top_classes=2),
             title="Demo edukasi: pneumonia pada X-ray dada", description=PERINGATAN, flagging_mode="never").launch(share=False)
''')
(OUT / "requirements.txt").write_text(f"numpy\npillow\nonnxruntime=={ort.__version__}\ngradio=={gr.__version__}\n")
r32 = T132[(T132.model == PILIH_DEMO) & (T132.versi == "FP32")].iloc[0]; lat = T133[(T133.model == PILIH_DEMO) & (T133.runtime == "ONNX Runtime FP32")]
(OUT / "model_card.md").write_text(f'''# Kartu Model — {PILIH_DEMO} PneumoniaMNIST (demo edukasi Bab 13)
**Penggunaan yang dimaksud:** pembelajaran dan riset; **bukan** untuk diagnosis, triase, atau keputusan klinis.
**Data:** PneumoniaMNIST {UK} px (MedMNIST v2, CC BY 4.0; sumber Kermany dkk., 2018 — X-ray dada anak, satu pusat). Pembagian resmi train/val/test.
**Arsitektur:** {PILIH_DEMO}, pra-latih ImageNet, keluaran 1 logit (pneumonia). Masukan: 1×{UK}×{UK}, nilai 0–1.
**Kinerja (set uji internal, n={len(yt)}):** AUC FP32 = {r32.AUC_uji:.3f}. Belum ada validasi eksternal.
**Latensi CPU (ONNX FP32, batch 1):** {"; ".join(f"{int(t)} thread: {m:.1f} ms" for t, m in zip(lat.thread, lat.ms_median))} — diukur pada CPU host Colab (spesifikasi di info_lingkungan.txt).
**Keterbatasan:** satu sumber data; anak-anak saja; resolusi rendah; tidak dikalibrasi untuk prevalensi lain; rentan pergeseran distribusi dan *shortcut* (lihat Bab 11–12).
**Risiko & mitigasi (konsep ISO 14971):** salah negatif → bukan pengganti pembacaan radiolog; demo memuat peringatan; tidak menyimpan citra unggahan.
**Keamanan:** demo dijalankan `share=False`; jangan unggah data pasien yang dapat dikenali.
**Versi:** {json.dumps(VERSI)}
''')
print((OUT / "model_card.md").read_text()[:1500])

## 13.7 Ringkasan, zip hasil, dan total waktu

In [ ]:
def bersih(o):
    if isinstance(o, dict): return {str(k): bersih(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [bersih(v) for v in o]
    if isinstance(o, (np.floating, float)): return round(float(o), 6)
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, np.bool_): return bool(o)
    return o
WAKTU["total_notebook"] = round(time.perf_counter() - T_MULAI, 2)
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada GPU (CPU)"
RINGKASAN = bersih(dict(smoke=SMOKE, gpu=GPU, versi=VERSI, waktu_detik=WAKTU, **(dict(ukuran_px=UK, ekspor=T131.to_dict("records"), kuantisasi=T132.to_dict("records"), latensi=T133.to_dict("records"), model_demo=PILIH_DEMO))))
(OUT / "ringkasan_bab13.json").write_text(json.dumps(RINGKASAN, indent=1, ensure_ascii=False))
pd.Series(WAKTU, name="detik").to_csv(OUT / "waktu_sel.csv")
arsip = shutil.make_archive(str(OUT), "zip", root_dir=OUT)
print("Berkas hasil:"); [print(" -", p.name, f"{p.stat().st_size/1e3:.0f} kB") for p in sorted(OUT.iterdir())]
print("ZIP:", arsip, f"{Path(arsip).stat().st_size/1e6:.2f} MB")
print(json.dumps(WAKTU, indent=1))
print(f"TOTAL WAKTU NOTEBOOK: {WAKTU['total_notebook']:.1f} s ({WAKTU['total_notebook']/60:.1f} menit) | GPU: {GPU}")
try:
    from google.colab import files  # unduh otomatis bila di Colab
    files.download(arsip)
except Exception:
    pass